# 01 — Data Discovery

Use this notebook to inspect the official portal services and print the exact layer names before downloading anything.

The datasets used in this project are:
- Administrative boundaries of Spain (IGN / CNIG)
- Healthcare resources WFS (sigMayores)
- Social resources WFS (sigMayores)
- Population by sex and age (INE)

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import geopandas as gpd
import numpy as np

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from scripts.config import (
    RAW_DIR, PROCESSED_DIR, OUTPUT_DIR,
    BOUNDARIES_WFS, HEALTH_WFS, SOCIAL_WFS, POPULATION_CSV,
    METRIC_CRS, MAP_CRS, GEOGRAPHIC_CRS,
    BOUNDARY_LAYER, HEALTH_LAYER, SOCIAL_LAYER,
    MUNICIPALITY_CODE_COL, MUNICIPALITY_NAME_COL, TOTAL_POP_COL,
)
from scripts.data_sources import SOURCES
from scripts.wfs_utils import discover_wfs_layers, load_wfs_layer, download_csv, save_geodataframe, save_dataframe
from scripts.population_utils import normalize_columns, build_population_65_plus
from scripts.analysis_utils import standardize_geodataframes, build_vulnerability_index, spatial_autocorrelation, top_ranked
from scripts.plotting_utils import save_choropleth
from scripts.export_utils import export_geodataframe, export_dataframe

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
for source in SOURCES:
    print(f"{source.kind}: {source.name}")
    print(source.url)
    print(source.notes)
    print("-" * 80)

In [ ]:
print("BOUNDARIES WFS layers:")
boundary_layers = discover_wfs_layers(BOUNDARIES_WFS)
for layer in boundary_layers:
    print(" -", layer)

print("\nHEALTH WFS layers:")
health_layers = discover_wfs_layers(HEALTH_WFS)
for layer in health_layers:
    print(" -", layer)

print("\nSOCIAL WFS layers:")
social_layers = discover_wfs_layers(SOCIAL_WFS)
for layer in social_layers:
    print(" -", layer)

In [ ]:
layer_catalog = {
    "boundary_layers": boundary_layers,
    "health_layers": health_layers,
    "social_layers": social_layers,
}
(PROCESSED_DIR / "layer_catalog.json").write_text(json.dumps(layer_catalog, indent=2), encoding="utf-8")
print("Saved layer catalog to data/processed/layer_catalog.json")

In [ ]:
# After you inspect the printed names, set the correct values in scripts/config.py
# Then restart the kernel and continue with Notebook 02.